In [1]:
import cv2
import pickle
import numpy as np
import os
import csv
import time

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from datetime import datetime
from win32com.client import Dispatch

In [3]:
def speak(str1):
    speak=Dispatch(("SAPI.SpVoice"))
    speak.Speak(str1)

In [10]:
video=cv2.VideoCapture(0)
facesdetect=cv2.CascadeClassifier('Data/haarcascade_frontalface_default.xml')

with open('Data/names.pkl','rb') as f:
    LABELS=pickle.load(f)
with open('Data/faces_data.pkl','rb') as f:
    FACES=pickle.load(f)

knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(FACES,LABELS)

COL_NAMES=['NAME' , 'TIME']

while True:
    ret,frame=video.read()
    gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    faces=facesdetect.detectMultiScale(gray,1.3,5)
    for (x,y,w,h) in faces:
        crop_img=frame[y:y+h,x:x+w]
        resized_img=cv2.resize(crop_img,(50,50)).flatten().reshape(1,-1)
        output=knn.predict(resized_img)

        ts=time.time()
        date=datetime.fromtimestamp(ts).strftime("%d-%m-%Y")
        timestamp=datetime.fromtimestamp(ts).strftime("%H:%M:%S")
        exist=os.path.isfile("Attendance/Attendance_"+date+".csv")

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,0,255),1)
        cv2.rectangle(frame,(x,y),(x+w,y+h),(50,50,255),2)
        cv2.rectangle(frame,(x,y-40),(x+w,y),(50,50,255),-1)
        cv2.putText(frame,str(output[0]),(x,y-15),cv2.FONT_HERSHEY_COMPLEX,1,(255,255,255),1)
        cv2.rectangle(frame,(x,y),(x+w,y+h),(50,50,255),1)

        attendance=[str(output[0]),str(timestamp)]

    cv2.imshow('Face Recognition',frame)

    if cv2.getWindowProperty('Face Recognition', cv2.WND_PROP_VISIBLE) < 1:
        break
    
    k = cv2.waitKey(1) & 0xFF

    if k==ord('a'):
        speak("Attendance Taken...")
        time.sleep(5)
        if exist:
            with open("Attendance/Attendance_"+date+".csv","+a") as csvfile:
                writer=csv.writer(csvfile)
                writer.writerow(attendance)
            csvfile.close()
        else:
            with open("Attendance/Attendance_"+date+".csv","+a") as csvfile:
                writer=csv.writer(csvfile)
                writer.writerow(COL_NAMES)
                writer.writerow(attendance)
            csvfile.close()

    if k==ord('q'): 
        break
    try:
        if cv2.getWindowProperty('Face Recognition', cv2.WND_PROP_VISIBLE) < 1:
            break
    except:
        break

video.release()
cv2.destroyAllWindows()